<a href="https://colab.research.google.com/github/dhiyasalmas/Machine_Learning/blob/main/MINDSTEM_ChatBot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Pelatihan Pengembangan AI - MINDSTEM (Part 2 - Chatbot)

Pada notebook ini, kita akan membuat sebuah aplikasi chatbot sederhana menggunakan model dari [HuggingFace](https://huggingface.co/) (atau [HF](https://huggingface.co/)). HF adalah platform untuk membangun dan berbagi model open-source AI. Kita akan menggunakan Google Colab untuk membuat aplikasi ini dan men-deploy prototype-nya menggunakan [Gradio](https://www.gradio.app/).


# Langkah 1: Instalasi

Kita akan menggunakan beberapa library berikut:

- `gradio` untuk membangun aplikasi web.
- `huggingface_hub` untuk memuat model dari HF.

In [ ]:
!pip install huggingface_hub
!pip install gradio

# Langkah 2: Mengakses LLM

Kita akan menggunakan inference endpoint dari model di [HF] langsung (tanpa memuatnya di memori, karena ukurannya sangat berat).

Untuk mempersingkat waktu, saya akan bagian temporary `HF_TOKEN` untuk pelatihan ini. Token ini hanya berlaku selama beberapa menit ke depan.

Dengan token ini, Anda bisa memuat berbagai [model inferensi di HF](https://huggingface.co/models?pipeline_tag=text-generation&sort=trending).


# Langkah 3: Membuat user interface dengan Gradio


Kita akan menggunakan fungsi dan prompt sederhana

```python
import os
import gradio as gr
from huggingface_hub import InferenceClient
from functools import partial
from google.colab import userdata

system_prompt="Kamu adalah asisten yang handal dan pakai bahasa Indonesia yang santun tapi santai (tidak baku)."
```


Fungsi `inference` utama yang akan kita pakai adalah sbb:
```python
def inference(prompt, token_hf, model, model_name):
  messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": prompt}]

  if token_hf is None or not token_hf.strip():
    token_hf = os.getenv("HF_TOKEN")
  
  client = InferenceClient(model=model, token=token_hf)
  tokens = f"**`{model_name}`**\n\n"
  for completion in client.chat_completion(messages, stream=True, max_tokens=200):
    tokens += completion.choices[0].delta.content
    yield tokens
```


In [ ]:
import os
import gradio as gr
from huggingface_hub import InferenceClient
from functools import partial
from google.colab import userdata

system_prompt="""
Kamu adalah asisten yang handal dan pakai bahasa Indonesia yang santun tapi santai (tidak baku).
"""

def inference(prompt, token_hf, model, model_name):
  messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": prompt}]

  if token_hf is None or not token_hf.strip():
    token_hf = userdata.get('HF_TOKEN')

  client = InferenceClient(model=model, token=token_hf)
  tokens = f"**`{model_name}`**\n\n"
  for completion in client.chat_completion(messages, stream=True, max_tokens=200):
    tokens += completion.choices[0].delta.content
    yield tokens


def hide_textbox():
  return gr.Textbox(visible=False)

with gr.Blocks(theme='ParityError/Interstellar') as demo:
    gr.Markdown("<center><h1>Chat Bot MINDSTEM</h1></center>")
    gr.Markdown("Deskripsi: Aplikasi untuk chat bot sederhana dalam bahasa Indonesia.")

    prompt = gr.Textbox(label="Prompt")
    token = gr.Textbox(label="Token HF", type="password", placeholder="Masukkan token HF Anda (bisa didapatkan dari akun Hugging Face)")


    with gr.Row():
      generate_btn = gr.Button("Kirim", variant="primary")

    with gr.Row():
      llm_output = gr.Markdown("LLama 3-70B Instruct")

    gr.on(
        triggers = [prompt.submit, generate_btn.click],
        fn = partial(inference, model="meta-llama/Llama-3.3-70b-Instruct", model_name="Llama 3.3-70B Instruct"),
        inputs=[prompt, token],
        outputs=[llm_output],
        show_progress='hidden'
    ).then(
        fn = hide_textbox,
        inputs = None,
        outputs = token
    )


demo.launch()

Running Gradio in a Colab notebook requires sharing enabled. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://63d82a439fd13d7821.gradio.live

This share link expires in 72 hours. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


# Setelah Pelatihan Ini


Ada banyak model dan aplikais yang bisa di-explore di [HuggingFace](https://huggingface.co/). Silakan explore dan share dengan MINDSTEM!